In [ ]:
pip install git+https://github.com/fluxnet/shuttle.git

In [ ]:
from pathlib import Path
import pandas as pd
import subprocess
import glob
import os

# -----------------------------
# Settings
# -----------------------------
#out_dir = Path(os.environ.get("PAPER_ROOT", ".") + "/fluxnet_2017_2025_V02")
# for Daniel
out_dir = Path(os.environ.get("PAPER_ROOT", ".") + "/Flux4Daniel")
out_dir.mkdir(parents=True, exist_ok=True)

target_igbp = ["CSH", "OSH", "SAV", "WSA", "WET", "MF", "ENF", "EBF", "DNF", "DBF", "CRO", "CVM", "GRA", "BSV" ]

#start_year = 2017
#end_year = 2025

# Change this to 1, 3, 4, etc.
# Use 1 if you want any data between 2017 and 2025.
#min_continuous_years = 1

os.chdir(out_dir)
print("Working in:", out_dir)

In [ ]:
import glob
import os

# -----------------------------
# Settings
# -----------------------------
out_dir = Path(os.environ.get("PAPER_ROOT", ".") + "/fluxnet_2017_2025_V02")
out_dir.mkdir(parents=True, exist_ok=True)

target_igbp = ["CSH", "OSH", "SAV", "WSA", "WET", "MF", "ENF", "EBF", "DNF", "DBF"]

start_year = 2017
end_year = 2025

# Change this to 1, 3, 4, etc.
# Use 1 if you want any data between 2017 and 2025.
min_continuous_years = 1

os.chdir(out_dir)
print("Working in:", out_dir)

In [ ]:
# -----------------------------
# Step 1: create latest Shuttle snapshot
# -----------------------------
subprocess.run(
    ["fluxnet-shuttle", "--verbose", "listall"],
    check=True
)

snapshots = sorted(out_dir.glob("fluxnet_shuttle_snapshot*.csv"))
snapshot_file = snapshots[-1]

snapshot = pd.read_csv(snapshot_file)

print("Using snapshot:", snapshot_file)
print("Snapshot rows:", len(snapshot))
print(snapshot.columns.tolist())

In [ ]:
# -----------------------------
# Step 2: filter by IGBP and years
# -----------------------------
metadata = snapshot.copy()

metadata["igbp"] = metadata["igbp"].astype(str).str.strip()
metadata["site_id"] = metadata["site_id"].astype(str).str.strip()

metadata = metadata[
    metadata["igbp"].isin(target_igbp)
].copy()

### for Daniel comment this part out
# Keep sites that overlap with 2017–2025
#metadata = metadata[
#    (metadata["first_year"] <= end_year) &
#    (metadata["last_year"] >= start_year)
#].copy()

metadata = metadata.drop_duplicates(subset="site_id")

site_list = metadata["site_id"].tolist()

print("Sites after IGBP filter:", len(metadata))

print("Number of sites to download:", len(site_list))

metadata[["site_id", "site_name", "igbp", "first_year", "last_year"]].head()

In [ ]:
metadata_file = out_dir / "selected_site_metadata.csv"
metadata.to_csv(metadata_file, index=False)

print("Saved metadata:", metadata_file)

In [ ]:
# -----------------------------
# Step 3: build site-year availability table
# -----------------------------
availability = []

for _, row in metadata.iterrows():
    available_start = max(int(row["first_year"]), start_year)
    available_end = min(int(row["last_year"]), end_year)
    
    for year in range(available_start, available_end + 1):
        availability.append({
            "site_id": row["site_id"],
            "igbp": row["igbp"],
            "year": year
        })

availability_df = pd.DataFrame(availability)

print("Availability rows:", len(availability_df))
availability_df.head()

In [ ]:
# -----------------------------
# Step 4: filter by continuous years
# -----------------------------
def max_consecutive_years(years):
    years = sorted(set(years))
    if not years:
        return 0
    
    max_run = 1
    current_run = 1
    
    for i in range(1, len(years)):
        if years[i] == years[i - 1] + 1:
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 1
            
    return max_run

site_continuity = (
    availability_df
    .groupby("site_id")["year"]
    .apply(max_consecutive_years)
    .reset_index(name="max_consecutive_years_2017_2025")
)

selected_sites_df = site_continuity[
    site_continuity["max_consecutive_years_2017_2025"] >= min_continuous_years
].copy()

selected_metadata = metadata.merge(
    selected_sites_df,
    on="site_id",
    how="inner"
)

site_list = selected_metadata["site_id"].tolist()

print(f"Selected sites with at least {min_continuous_years} continuous years:", len(site_list))

selected_metadata[
    ["site_id", "site_name", "igbp", "first_year", "last_year", "max_consecutive_years_2017_2025"]
].head()

In [ ]:
# -----------------------------
# Step 5: save selected metadata before downloading
# -----------------------------
selected_metadata_file = out_dir / f"selected_sites_igbp_{start_year}_{end_year}_min{min_continuous_years}yr.csv"
availability_file = out_dir / f"selected_sites_availability_{start_year}_{end_year}_min{min_continuous_years}yr.csv"

selected_metadata.to_csv(selected_metadata_file, index=False)
availability_df.to_csv(availability_file, index=False)

print("Saved:")
print(selected_metadata_file)
print(availability_file)

In [ ]:
# -----------------------------
# Step 6: download only selected sites
# -----------------------------
cmd = [
    "fluxnet-shuttle",
    "--verbose",
    "download",
    "-f", str(snapshot_file),
    "-o", str(out_dir),
    "-s", *site_list,
]

result = subprocess.run(
    cmd,
    input="\n" * 20,
    text=True,
    check=True
)

print("Download finished.")